# FP32 vs INT8 epoch 3 — trực quan 100 ảnh
Notebook tạo ảnh ba cột Ground Truth, FP32 và INT8 rồi upload kết quả thành Kaggle Dataset riêng. Không chạy lại benchmark chỉ số.

In [ ]:
import os, subprocess, sys
from pathlib import Path
import torch
REPO = Path('/kaggle/working/EchteAI')
if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/NguyenDucThang-tb/EchteAI.git', str(REPO)], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[coco]', 'psutil', 'kagglehub'], check=True)
import kagglehub, yaml
print('Repository:', Path.cwd())

## 1. Tải dữ liệu và checkpoint

In [ ]:
KAGGLE_USERNAME = 'nguyenducthangtb'
SEADRONESSEE_DATASET = 'ubiratanfilho/sds-dataset'
FP32_DATASET = f'{KAGGLE_USERNAME}/echteai-seadronessee-m3-checkpoints'
QAT_DATASET = f'{KAGGLE_USERNAME}/echteai-seadronessee-m3-qat-epoch3'
RESULT_DATASET = f'{KAGGLE_USERNAME}/echteai-fp32-int8-visualizations-epoch3-v2'

def find_dataset_root(start):
    start = Path(start)
    for candidate in [start, *[p.parent.parent for p in start.rglob('instances_train.json')]]:
        if (candidate/'annotations/instances_train.json').exists() and (candidate/'images/val').is_dir():
            return candidate
    raise FileNotFoundError(f'Không tìm thấy SeaDronesSee trong {start}')

DATA_ROOT = find_dataset_root(kagglehub.dataset_download(SEADRONESSEE_DATASET))
FP32_ROOT = Path(kagglehub.dataset_download(FP32_DATASET, force_download=True))
QAT_ROOT = Path(kagglehub.dataset_download(QAT_DATASET, force_download=True))
fp32_matches = list(FP32_ROOT.rglob('fp32_best.pt'))
qat_matches = list(QAT_ROOT.rglob('qat_last.pt'))
assert fp32_matches, FP32_ROOT
assert qat_matches, QAT_ROOT
FP32_CHECKPOINT = fp32_matches[0]
QAT_CHECKPOINT = qat_matches[0]
qat_payload = torch.load(QAT_CHECKPOINT, map_location='cpu', weights_only=False)
assert int(qat_payload.get('epoch', -1)) == 3, qat_payload.get('epoch')
print('Data:', DATA_ROOT)
print('FP32:', FP32_CHECKPOINT)
print('QAT epoch:', qat_payload.get('epoch'), QAT_CHECKPOINT)

## 2. Tạo runtime config và convert QAT epoch 3 sang INT8

In [ ]:
OUTPUT = Path('/kaggle/working/fp32_int8_visual_epoch3')
OUTPUT.mkdir(parents=True, exist_ok=True)
config = yaml.safe_load(Path('configs/seadronessee_colab.yaml').read_text())
config['dataset'].update({
    'train_images': str(DATA_ROOT/'images/train'), 'train_annotations': str(DATA_ROOT/'annotations/instances_train.json'),
    'val_images': str(DATA_ROOT/'images/val'), 'val_annotations': str(DATA_ROOT/'annotations/instances_val.json'),
    'test_images': str(DATA_ROOT/'images/val'), 'test_annotations': str(DATA_ROOT/'annotations/instances_val.json'),
})
config['dataset']['workers'] = 2
config['model']['pretrained_backbone'] = False
config['output']['directory'] = str(OUTPUT)
RUNTIME_CONFIG = OUTPUT/'runtime.yaml'
RUNTIME_CONFIG.write_text(yaml.safe_dump(config, sort_keys=False))
INT8_CHECKPOINT = OUTPUT/'selective_int8_epoch3.pt'
print('Runtime config:', RUNTIME_CONFIG)

In [ ]:
sys.path.insert(0, str(REPO/'Python'))
from pipelines.convnext_qat.config import load_config, quantized_modules_for_variant
from pipelines.convnext_qat.models import build_fasterrcnn_convnext
from pipelines.convnext_qat.checkpoint import load_checkpoint, save_checkpoint
from pipelines.convnext_qat.quantization import prepare_selective_qat, convert_selective_qat
runtime = load_config(RUNTIME_CONFIG)
metadata = qat_payload.get('extra', {})
variant = metadata.get('variant', 'M3')
backend = metadata.get('backend', 'x86')
modules = metadata.get('quantized_modules', quantized_modules_for_variant(runtime, variant))
model = build_fasterrcnn_convnext(runtime)
qat_model = prepare_selective_qat(model, variant, backend, quantized_modules=modules)
load_checkpoint(QAT_CHECKPOINT, qat_model, map_location='cpu')
int8_model = convert_selective_qat(qat_model.cpu()).eval()
save_checkpoint(INT8_CHECKPOINT, int8_model, epoch=3, metrics={'source_qat_epoch': 3}, extra={
    'variant': variant, 'backend': backend, 'format': 'selective_int8', 'quantized_modules': modules,
})
print('INT8:', INT8_CHECKPOINT, f'{INT8_CHECKPOINT.stat().st_size/2**20:.2f} MB')

## 3. Tạo 100 ảnh Ground Truth | FP32 | INT8

In [ ]:
VISUAL_DIR = OUTPUT/'visualizations'
command = [sys.executable, '-u', 'scripts/visualize_fp32_int8.py',
    '--config', str(RUNTIME_CONFIG), '--fp32-checkpoint', str(FP32_CHECKPOINT),
    '--int8-checkpoint', str(INT8_CHECKPOINT), '--images', '100',
    '--score-threshold', '0.5', '--threads', '1', '--output-dir', str(VISUAL_DIR)]
print('Command:', ' '.join(command), flush=True)
subprocess.run(command, check=True, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
from IPython.display import display, Image as DisplayImage
images = sorted((VISUAL_DIR/'images').glob('*.jpg'))
assert len(images) == 100, len(images)
print('Created visualizations:', len(images))
display(*[DisplayImage(filename=str(path), width=1200) for path in images[:3]])

## 4. Upload toàn bộ kết quả thành Kaggle Dataset

In [ ]:
for path in sorted(OUTPUT.rglob('*')):
    if path.is_file(): print(path.relative_to(OUTPUT), f'{path.stat().st_size/2**20:.2f} MB')
kagglehub.dataset_upload(
    RESULT_DATASET, str(OUTPUT),
    version_notes='100 Ground Truth, FP32 and selective INT8 epoch 3 visualizations',
)
print('Uploaded:', f'https://www.kaggle.com/datasets/{RESULT_DATASET}')